# Bloque 1 — Análisis Exploratorio con PySpark
## Dataset: SIVIGILA — Vigilancia en Salud Pública, Colombia 2019
### Parcial Final — Machine Learning con PySpark y Docker

**Autor:** Sergio Prieto  
**Fecha:** Mayo 2026  
**Profesora:** Luz Adriana Gutiérrez Rodríguez  

---
## Objetivo
Realizar un análisis exploratorio completo sobre datos reales del sistema de vigilancia epidemiológica colombiano (SIVIGILA), aplicando transformaciones PySpark, estadística descriptiva, detección de anomalías y visualizaciones.

## Dataset
El dataset contiene **205,532 registros** de notificaciones obligatorias de eventos de salud pública en Colombia durante 2019, con 69 tipos de eventos distintos reportados en 35 departamentos a lo largo de 52 semanas epidemiológicas.

**Fuente:** Instituto Nacional de Salud — SIVIGILA 2019


## Configuración del entorno
Se configura PySpark en modo local y se establece el HADOOP_HOME para compatibilidad con Windows.


In [ ]:
import os
import sys

# Fix para PySpark en Windows
HADOOP_HOME = os.path.join(os.environ.get("TEMP", os.path.expanduser("~")), "hadoop_tmp")
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.makedirs(os.path.join(HADOOP_HOME, "bin"), exist_ok=True)
winutils_path = os.path.join(HADOOP_HOME, "bin", "winutils.exe")
if not os.path.exists(winutils_path):
    with open(winutils_path, "wb") as f:
        f.write(b"")

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, sum as _sum, count, avg, stddev, min as _min, max as _max,
    percentile_approx, when, isnan, isnull, lit, regexp_replace, trim,
    round as spark_round, desc, asc,
)
from pyspark.sql.types import IntegerType, StringType
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

CSV_PATH = "../data/sivigila.csv"
OUTPUT_DIR = "../salidas"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Tarea 1 — Carga del dataset con PySpark
Se carga el CSV usando pandas (para evitar problemas de Hadoop en Windows) y se convierte a Spark DataFrame.  
Se muestran: esquema, conteo de registros y primeras filas.

**Transformaciones de limpieza:**
- `ANO`: el CSV usa formato "2.019" con punto de miles → se limpia a 2019
- `conteo_casos`, `SEMANA`, `COD_DPTO_O`, `COD_MUN_O`, `COD_EVE` → se convierten a `IntegerType`
- `nom_mun`: valores vacíos → `None`


In [ ]:
spark = (
    SparkSession.builder
    .appName("SIVIGILA_Bloque1_EDA")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Cargando datos...")
pdf_raw = pd.read_csv(CSV_PATH, dtype=str, encoding="utf-8", keep_default_na=False)
pdf_raw = pdf_raw.rename(columns={"ANO": "ANO_STR"})

df = spark.createDataFrame(pdf_raw)
df = (
    df
    .withColumn("conteo_casos", col("conteo_casos").cast(IntegerType()))
    .withColumn("SEMANA",       col("SEMANA").cast(IntegerType()))
    .withColumn("COD_DPTO_O",   col("COD_DPTO_O").cast(IntegerType()))
    .withColumn("COD_MUN_O",    col("COD_MUN_O").cast(IntegerType()))
    .withColumn("COD_EVE",      col("COD_EVE").cast(IntegerType()))
    .withColumn("ANO",
        regexp_replace(col("ANO_STR"), r"\.", "").cast(IntegerType()))
    .withColumn("nom_mun",
        when(trim(col("nom_mun")) == "", None).otherwise(trim(col("nom_mun"))))
    .drop("ANO_STR")
)
df = df.cache()
total_registros = df.count()

print(f"Registros totales: {total_registros:,}")
print(f"Columnas: {len(df.columns)}")
print("\n--- Esquema ---")
df.printSchema()
print("\n--- Primeras 10 filas ---")
df.show(10, truncate=False)


## Tarea 2 — Tres transformaciones analíticas

Se aplican tres transformaciones que responden preguntas concretas del dominio:

| Transformación | Pregunta analítica |
|---------------|-------------------|
| `filter` | ¿Cuáles son los departamentos con más casos de DENGUE? |
| `groupBy` | ¿Cuál es el total de casos por tipo de evento? |
| `withColumn` + `join` | ¿Qué departamentos tienen la mayor tasa de dengue por cada 1000 casos? |


In [ ]:
# T2a: FILTER — Top departamentos con mas DENGUE
print("--- T2a: Filter — Top 10 departamentos con mas casos de DENGUE ---")
df_dengue = df.filter(col("Nombre") == "DENGUE").cache()
df_dengue_dpto = (
    df_dengue.groupBy("COD_DPTO_O")
    .agg(_sum("conteo_casos").alias("total_casos_dengue"))
    .orderBy(desc("total_casos_dengue"))
)
df_dengue_dpto.show(10)

# T2b: GROUPBY — Total casos por evento
print("\n--- T2b: GroupBy — Total de casos por evento (top 15) ---")
df_evento_total = (
    df.groupBy("Nombre")
    .agg(_sum("conteo_casos").alias("total_casos"))
    .orderBy(desc("total_casos"))
)
df_evento_total.show(15, truncate=False)

# T2c: WITHCOLUMN + JOIN — Tasa de dengue por dpto (x1000)
print("\n--- T2c: WithColumn + Join — Tasa de dengue por dpto (x1000) ---")
total_por_dpto = df.groupBy("COD_DPTO_O").agg(_sum("conteo_casos").alias("total_dpto"))
df_tasa = (
    df_dengue_dpto.join(total_por_dpto, "COD_DPTO_O")
    .withColumn("tasa_dengue_x1000",
        spark_round(col("total_casos_dengue") / col("total_dpto") * 1000, 2))
    .orderBy(desc("tasa_dengue_x1000"))
)
df_tasa.show(10)


## Tarea 3 — Estadística descriptiva

Se calculan:
- `describe()` para variables numéricas (SEMANA, COD_DPTO_O, conteo_casos)
- `summary()` con percentiles 25%, 50%, 75%
- Percentiles 90, 95 y 99 de `conteo_casos`
- Conteos por categoría (Nombre del evento y semana)


In [ ]:
print("--- describe() ---")
df.describe(["SEMANA", "COD_DPTO_O", "conteo_casos"]).show()

print("\n--- summary() conteo_casos ---")
df.select("conteo_casos").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show()

print("\n--- Percentiles 90, 95, 99 ---")
df.select(
    percentile_approx("conteo_casos", 0.90).alias("p90"),
    percentile_approx("conteo_casos", 0.95).alias("p95"),
    percentile_approx("conteo_casos", 0.99).alias("p99"),
).show()

print("\n--- Top 10 eventos mas frecuentes ---")
df.groupBy("Nombre").count().orderBy(desc("count")).show(10, truncate=False)

print("\n--- Casos totales por semana ---")
df.groupBy("SEMANA").agg(_sum("conteo_casos").alias("casos_semana"))   .orderBy("SEMANA").show(52)


## Tarea 4 — Valores faltantes, duplicados y atípicos

**Metodología:**
- **Nulos:** conteo por columna en una sola agregación para eficiencia
- **Duplicados exactos:** `dropDuplicates()` sobre todas las columnas
- **Duplicados parciales:** groupBy por llave lógica (COD_EVE + SEMANA + COD_MUN_O)
- **Atípicos:** método IQR (Q3 + 1.5 × IQR) sobre `conteo_casos`
- **Datos anómalos:** registros con `COD_DPTO_O = 0` (desconocido)


In [ ]:
# Nulos en una sola pasada
print("--- Valores nulos por columna ---")
nulos_exprs = [_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
nulos_row = df.agg(*nulos_exprs).collect()[0]
for c in df.columns:
    n = nulos_row[c]
    print(f"  {c}: {n} nulos ({(n/total_registros)*100:.2f}%)")

# Duplicados
print("\n--- Duplicados exactos ---")
count_sin_dup = df.dropDuplicates().count()
print(f"  Duplicados exactos: {total_registros - count_sin_dup}")

# Duplicados parciales
print("\n--- Llaves duplicadas (COD_EVE+SEMANA+COD_MUN_O) ---")
dup_llave = (
    df.groupBy("COD_EVE", "SEMANA", "COD_MUN_O")
    .agg(count("*").alias("n"))
    .filter(col("n") > 1)
)
print(f"  Grupos con llave repetida: {dup_llave.count()}")

# Atipicos (IQR)
print("\n--- Valores atipicos en conteo_casos (IQR) ---")
stats = df.select(
    percentile_approx("conteo_casos", 0.25).alias("q1"),
    percentile_approx("conteo_casos", 0.75).alias("q3"),
).collect()[0]
q1, q3 = stats["q1"], stats["q3"]
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
print(f"  Q1={q1}, Q3={q3}, IQR={iqr}, Limite superior={lim_sup}")
atipicos = df.filter(col("conteo_casos") > lim_sup).count()
print(f"  Registros atipicos: {atipicos} ({(atipicos/total_registros)*100:.2f}%)")

print("\n  Top 10 valores mas altos de conteo_casos:")
df.select("Nombre", "nom_mun", "SEMANA", "conteo_casos") \
  .orderBy(desc("conteo_casos")).show(10, truncate=False)

print(f"\n  Registros con departamento desconocido (cod 0): {df.filter(col('COD_DPTO_O') == 0).count()}")


## Tarea 5 — Visualizaciones

Se construyen dos gráficos (convirtiendo a Pandas):

1. **Top 15 eventos por total de casos** (barras horizontales)
2. **Serie temporal de dengue por semana epidemiológica** (línea con media móvil)

Estos gráficos sustentan las conclusiones sobre concentración de morbilidad y estacionalidad.


In [ ]:
# --- Grafico 1: Top 15 eventos por total de casos ---
pdf_eventos = df_evento_total.limit(15).toPandas().sort_values("total_casos")

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(pdf_eventos["Nombre"], pdf_eventos["total_casos"], color="#1f77b4")
ax.set_xlabel("Total de casos reportados", fontsize=11)
ax.set_title("Top 15 eventos de salud publica por numero de casos\nSIVIGILA Colombia - 2019",
             fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, pdf_eventos["total_casos"]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_top15_eventos.png"), bbox_inches="tight")
plt.show()

# --- Grafico 2: Serie temporal de DENGUE por semana ---
pdf_dengue_semanal = (
    df_dengue.groupBy("SEMANA")
    .agg(_sum("conteo_casos").alias("casos_dengue"))
    .orderBy("SEMANA").toPandas()
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pdf_dengue_semanal["SEMANA"], pdf_dengue_semanal["casos_dengue"],
        marker="o", linewidth=1.5, markersize=3, color="#d62728")
ax.fill_between(pdf_dengue_semanal["SEMANA"], pdf_dengue_semanal["casos_dengue"],
                alpha=0.15, color="#d62728")
ax.set_xlabel("Semana epidemiologica", fontsize=11)
ax.set_ylabel("Casos de dengue", fontsize=11)
ax.set_title("Casos de dengue por semana epidemiologica\nColombia 2019",
             fontsize=13, fontweight="bold")
ax.set_xticks(range(1, 53, 4))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
z = pd.Series(pdf_dengue_semanal["casos_dengue"].values).rolling(4, center=True).mean()
ax.plot(pdf_dengue_semanal["SEMANA"], z, color="black", linewidth=2, linestyle="--",
        label="Tendencia (media movil 4 sem)")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_dengue_semanal.png"), bbox_inches="tight")
plt.show()


## Tarea 6 — Conclusiones cuantitativas

A continuación se calculan las métricas que sustentan las conclusiones y se redactan en formato Markdown.


In [ ]:
top3 = df_evento_total.limit(3).collect()
total_casos_global = df.agg(_sum("conteo_casos")).collect()[0][0]
top3_casos = sum(r["total_casos"] for r in top3)

dengue_pico = (
    df_dengue.groupBy("SEMANA")
    .agg(_sum("conteo_casos").alias("casos_dengue"))
    .orderBy(desc("casos_dengue")).first()
)

dpto_top = (
    df.groupBy("COD_DPTO_O")
    .agg(_sum("conteo_casos").alias("total"))
    .orderBy(desc("total")).first()
)

top10_dptos_sum = (
    df.groupBy("COD_DPTO_O")
    .agg(_sum("conteo_casos").alias("total"))
    .orderBy(desc("total")).limit(10)
    .agg(_sum("total")).collect()[0][0]
)

print(f"Total casos global: {total_casos_global:,}")
print(f"Top 3 eventos: {top3_casos:,} ({(top3_casos/total_casos_global)*100:.1f}%)")
print(f"Pico dengue: semana {dengue_pico['SEMANA']} con {dengue_pico['casos_dengue']:,} casos")
print(f"Dpto lider: {dpto_top['COD_DPTO_O']} con {dpto_top['total']:,} casos ({(dpto_top['total']/total_casos_global)*100:.1f}%)")
print(f"Top 10 dptos: {(top10_dptos_sum/total_casos_global)*100:.1f}% del total")

df.unpersist()
df_dengue.unpersist()
spark.stop()


---

## CONCLUSIONES DEL BLOQUE 1

### Conclusión 1: Concentración de morbilidad
Los tres eventos más reportados en Colombia durante 2019 fueron **Agresiones por animales transmisores de rabia**, **Dengue** y **VCM/VIF/VSX**. En conjunto representan aproximadamente el **49%** del total de casos notificados al SIVIGILA. Esto evidencia que unas pocas patologías concentran la mayoría de la carga de notificación obligatoria, lo cual tiene implicaciones para la asignación de recursos en vigilancia epidemiológica.

### Conclusión 2: Estacionalidad del dengue
El dengue presenta un patrón estacional claro: los casos se incrementan progresivamente desde el primer trimestre, alcanzan su **pico máximo hacia la mitad del año** (semanas 20-35), coincidiendo con la temporada de lluvias en gran parte del territorio colombiano, y descienden en el cuarto trimestre. Este hallazgo es consistente con la biología del vector *Aedes aegypti*.

### Conclusión 3: Concentración geográfica
La notificación de eventos de salud pública está fuertemente concentrada: **los 10 departamentos con mayor notificación acumulan más del 60% de los casos**. El departamento líder concentra más del 11% del total nacional. Esto puede reflejar tanto una mayor carga real de enfermedad como una mejor infraestructura de vigilancia en los departamentos más poblados.

**Dataset:** SIVIGILA 2019 — 205,532 registros procesados  
**Figuras:** `salidas/fig1_top15_eventos.png`, `salidas/fig2_dengue_semanal.png`
